# Spark X2.6-4B full fine-tune — direct one-shot Colab

**This file replaces the former gated all-modality launcher.** The old CPU-preflight/qualification notebook is archived as [`Spark_Formal_Omni_Gated.ipynb`](Spark_Formal_Omni_Gated.ipynb).

**One-shot by design.** Nothing is assumed to survive from a previous Colab VM. Run the cells once, in order, on a fresh GPU runtime. The notebook clones the GitHub source, installs one isolated CUDA environment, downloads the original Spark + SigLIP weights, builds a pinned public image/frame corpus, runs connector alignment, then runs full-parameter training. There is **no CPU runtime preflight**, no qualification gate, no instructor call, and no dependency on an earlier notebook or Drive mount.

**Hardware.** Use one CUDA GPU with **≥46 GiB** and native BF16; an RTX PRO 6000 Blackwell / G4 (98 GiB) is the tested target. The wrapper has **4,201,669,760 parameters**; formal full training uses FP32 parameter storage, BF16 AMP and AdamW8bit. A 40 GiB GPU is not enough.

**Data.** The default path downloads two public, pinned components of `TassP0lar/SparkX2.6-MM`:

- `releases/docci-training-source-v1` — 8,776 human DOCCI captions/images.
- `releases/full-ft-v1/vision-core` — 42,215 COCO/COCO-CN/controlled-geometry task variants over 7,192 media packets.

The builder flattens shared-media variants, verifies every TAR member, makes a deterministic source-group development split (2%), and writes **50,991 records / 17,120 images / 3.05 GB**. This is a public image/frame subset, not the frozen 79,706-record formal-v1 corpus and not a sealed-test benchmark. For the exact formal-v1 corpus, set `DATASET_SOURCE="local"` and point `DATASET_LOCAL_DIR` at an existing dataset root.

**Output.** The final complete optimizer checkpoint is uploaded to the private Hugging Face model repo in `HF_CHECKPOINT_REPO` when `PUSH_CHECKPOINT=True`. The upload is the only persistence step; keep the token in Colab Secrets, never in a cell.

**GitHub must contain the image/frame runtime.** The notebook hard-checks for `multimodal/`, `scripts/train_multimodal_studio.py`, and `colab/stage1/` including `prepare_public_image_frame.py`. The existing `Spark_Formal_Full_Finetune.ipynb` is a different, gated all-modality launcher and is not used here.

## 1. One-shot configuration

Defaults train the public image/frame corpus. Change `REPO_URL`/`REPO_REF` to the branch where you pushed this workspace. Changing the corpus or stage after a checkpoint exists creates a different training contract; this notebook starts fresh each run.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/BaiTian6641/Spark-X2.6-VL.git"
REPO_REF = "codex/formal-colab"  # branch or tag that contains the image/frame runtime
REPO_COMMIT = ""                  # optional exact 40-hex commit; overrides REPO_REF
WORK = Path("/content/spark-mm")

# "public_hf" builds the pinned DOCCI + vision-core image/frame corpus.
# "local" uses DATASET_LOCAL_DIR and requires the reviewed dataset layout.
DATASET_SOURCE = "public_hf"
DATASET_LOCAL_DIR = WORK / "datasets/multimodal_formal_v1"
DATASET_DIR = WORK / "datasets/public_image_frame"

MIN_GPU_GIB = 46
ALIGNMENT_SAMPLES = 1024
ALIGNMENT_STEPS = 256
ALIGNMENT_ACCUM = 4
ALIGNMENT_LR = 1e-4
FULL_ACCUM = 8
FULL_LR = 1e-5
MAX_LENGTH = 2048
POOL_GRID = 16
SAVE_EVERY = 250
DEV_EVERY = 500
EVAL_LIMIT = 32
WALL_HOURS = 9.0

RUN_ID = "spark-mm-oneshot-" + time.strftime("%Y%m%d-%H%M%S", time.gmtime())
PUSH_CHECKPOINT = True
HF_CHECKPOINT_REPO = "TassP0lar/SparkX2.6-MM-checkpoints"

REPO = WORK / "repo"
ALIGNMENT_ID = RUN_ID + "-alignment"
FULL_ID = RUN_ID + "-full"
ALIGNMENT_CKPT = None
FINAL_CKPT = None
HF_TOKEN = None

WORK.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    "run_id": RUN_ID, "work": str(WORK), "dataset_source": DATASET_SOURCE,
    "gpu_min_gib": MIN_GPU_GIB, "wall_hours": WALL_HOURS,
    "alignment": [ALIGNMENT_SAMPLES, ALIGNMENT_STEPS, ALIGNMENT_ACCUM],
    "full": [FULL_ACCUM, FULL_LR, WALL_HOURS],
}, indent=2))

## 2. GPU contract — fast, no CPU preflight

This only checks the assigned accelerator. It does not validate the dataset, download models, or run a training gate.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Runtime > Change runtime type, then rerun.")
if torch.version.hip:
    raise RuntimeError("This notebook requires the CUDA build, not the historical ROCm build.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The assigned GPU does not report native BF16 support.")

props = torch.cuda.get_device_properties(0)
gpu_gib = props.total_memory / 2**30
print(json.dumps({
    "gpu": props.name,
    "total_gib": round(gpu_gib, 2),
    "capability": list(torch.cuda.get_device_capability(0)),
    "bf16": True,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
}, indent=2))
if gpu_gib < MIN_GPU_GIB:
    raise RuntimeError(
        f"Full-parameter Spark needs at least {MIN_GPU_GIB} GiB on one GPU; "
        f"this assignment reports {gpu_gib:.2f} GiB. Use a G4/RTX PRO 6000 or larger."
    )

## 3. Clone the GitHub source

A fresh clone every run is intentional. If a required path is missing, commit and push the image/frame files listed in the error before opening this notebook again.

In [ ]:
def run(command, cwd=None, env=None, check=True):
    print("$", " ".join(str(part) for part in command), flush=True)
    return subprocess.run([str(part) for part in command], cwd=None if cwd is None else str(cwd),
                          env=env, check=check, text=True)

if REPO.exists():
    shutil.rmtree(REPO)
REPO.parent.mkdir(parents=True, exist_ok=True)
if REPO_COMMIT:
    REPO.mkdir()
    run(["git", "init"], cwd=REPO)
    run(["git", "remote", "add", "origin", REPO_URL], cwd=REPO)
    run(["git", "fetch", "--depth", "1", "origin", REPO_COMMIT], cwd=REPO)
    run(["git", "checkout", "--force", "FETCH_HEAD"], cwd=REPO)
else:
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO])

required = [
    "multimodal/model.py",
    "multimodal/collator.py",
    "multimodal/data.py",
    "scripts/train_multimodal_studio.py",
    "scripts/prepare_vision_component.py",
    "colab/stage1/environment.py",
    "colab/stage1/environment_spec.json",
    "colab/stage1/candidate-cu130-py313.lock",
    "colab/stage1/prepare_models.py",
    "colab/stage1/port_trainer.py",
    "colab/stage1/prepare_public_image_frame.py",
]
missing = [path for path in required if not (REPO / path).is_file()]
if missing:
    raise RuntimeError(
        "The GitHub checkout lacks the image/frame runtime. Commit and push these paths, then rerun:\n  "
        + "\n  ".join(missing)
    )
sys.path.insert(0, str(REPO))
head = run(["git", "rev-parse", "HEAD"], cwd=REPO).stdout.strip()
print("SOURCE_COMMIT", head)

## 4. Install the isolated CUDA environment once

This is package installation, not a CPU preflight. It creates `/content/spark-mm/venv` from the reviewed hash-pinned CUDA lock and verifies `pip check`. The Colab kernel packages are not replaced.

In [ ]:
from colab.stage1.environment import bootstrap, child_environment

receipt = bootstrap(REPO, WORK)
PYTHON = WORK / "venv/bin/python"
if not PYTHON.is_file():
    raise RuntimeError("The isolated environment did not produce venv/bin/python")
TRAIN_ENV = child_environment(REPO, WORK)
TRAIN_ENV["PYTHONPATH"] = str(REPO)
print(json.dumps({"python": str(PYTHON), "environment": receipt}, indent=2))

## 5. Download original pretrained Spark and SigLIP

No workstation checkpoint is uploaded or used. Spark comes from the pinned `XHToken/Spark-X2.6-4B` revision; SigLIP is the pinned `google/siglip2-base-patch16-256`, reduced to its headless vision tower.

In [ ]:
env = child_environment(REPO, WORK)
run([PYTHON, "-m", "colab.stage1.prepare_models", "--project", REPO], cwd=REPO, env=env)
run([PYTHON, "scripts/prepare_vision_component.py", "--local-files-only"], cwd=REPO, env=env)
for path in [
    REPO / "models/Spark-X2.6-4B/config.json",
    REPO / "models/SigLIP2-base-256/vision-only/config.json",
    REPO / "evidence/Spark-X2.6-4B-source.json",
    REPO / "evidence/multimodal-s0-vision-source.json",
]:
    if not path.is_file():
        raise RuntimeError("Missing prepared model artifact: " + str(path))
vision = json.loads((REPO / "evidence/multimodal-s0-vision-source.json").read_text())
print(json.dumps({
    "vision_parameters": vision["derived_artifact"]["parameter_count"],
    "patch_grid": vision["derived_artifact"]["patch_grid"],
    "dense_shape": vision["derived_artifact"]["dense_patch_output_shape"],
}, indent=2))

## 6. Build the pinned public image/frame corpus

Downloads 3.05 GB of media from the pinned public dataset revision and verifies each TAR member. The 2% development split is source-group isolated. The sealed test split is not created or touched.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

if DATASET_SOURCE == "public_hf":
    valid = False
    if (DATASET_DIR / "BUILD.json").is_file():
        try:
            prior = json.loads((DATASET_DIR / "BUILD.json").read_text())
            valid = bool(prior.get("validation", {}).get("valid"))
        except Exception:
            valid = False
    if not valid:
        if DATASET_DIR.exists():
            shutil.rmtree(DATASET_DIR)
        env = child_environment(REPO, WORK)
        run([PYTHON, "-m", "colab.stage1.prepare_public_image_frame",
             "--cache", WORK / "public-cache", "--output", DATASET_DIR], cwd=REPO, env=env)
    if WORK.joinpath("public-cache").exists():
        shutil.rmtree(WORK / "public-cache")
elif DATASET_SOURCE == "local":
    DATASET_DIR = Path(DATASET_LOCAL_DIR).resolve()
    if not (DATASET_DIR / "dataset_manifest.json").is_file():
        raise RuntimeError("DATASET_SOURCE='local' requires DATASET_LOCAL_DIR with dataset_manifest.json")
else:
    raise ValueError("DATASET_SOURCE must be 'public_hf' or 'local'")

manifest = json.loads((DATASET_DIR / "dataset_manifest.json").read_text())
validation_path = WORK / "dataset_validation.json"
validation_env = child_environment(REPO, WORK)
validation_env.update({"PYTHONPATH": str(REPO), "DATASET_DIR": str(DATASET_DIR),
                       "VALIDATION_OUT": str(validation_path)})
validation_code = (
    "import json, os; from pathlib import Path; from multimodal.data import validate_dataset; "
    "v=validate_dataset(os.environ['DATASET_DIR'], splits=('train','dev')); "
    "Path(os.environ['VALIDATION_OUT']).write_text(json.dumps(v)); "
    "raise SystemExit(0 if v.get('valid') else 1)"
)
run([PYTHON, "-c", validation_code], cwd=REPO, env=validation_env)
validation = json.loads(validation_path.read_text())
if not validation.get("valid"):
    raise RuntimeError("Dataset validation failed: " + "; ".join(validation.get("errors", [])[:5]))
DATASET_MANIFEST_SHA = sha256_file(DATASET_DIR / "dataset_manifest.json")
print(json.dumps({
    "dataset": str(DATASET_DIR),
    "records": manifest["record_count"],
    "splits": manifest["by_split"],
    "tasks": manifest["by_task"],
    "languages": manifest["by_language"],
    "media_files": validation["unique_media_files"],
    "dataset_manifest_sha256": DATASET_MANIFEST_SHA,
}, indent=2))

# Training is local-only after this point.
TRAIN_ENV = child_environment(REPO, WORK)
TRAIN_ENV["PYTHONPATH"] = str(REPO)
TRAIN_ENV.update({
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "TOKENIZERS_PARALLELISM": "false",
})

## 7. Generate the CUDA entrypoint

The historical `scripts/train_multimodal_studio.py` is ROCm-only. `port_trainer.py` creates `scripts/train_multimodal_cuda.py` without editing the historical source. This notebook passes a generic CUDA memory/BF16 guard; the reviewed G4 name check is not required.

In [ ]:
from colab.stage1.port_trainer import materialize

port = materialize(REPO, minimum_gpu_gib=MIN_GPU_GIB, require_reviewed_gpu=False)
TRAIN = REPO / "scripts/train_multimodal_cuda.py"
if not TRAIN.is_file():
    raise RuntimeError("CUDA entrypoint was not created")
print(json.dumps(port, indent=2))

## 8. Connector alignment — 1,024 real captions, 256 updates

Spark and SigLIP are frozen; only the 3.747M media/position connectors train. This is the same stage shape as the workstation run, adapted to the public corpus.

In [ ]:
if ALIGNMENT_CKPT is None:
    align_dir = REPO / "results" / ALIGNMENT_ID
    if align_dir.exists():
        raise RuntimeError(f"Alignment run already exists: {align_dir}. Use a new RUN_ID.")
    command = [
        PYTHON, TRAIN,
        "--run-id", ALIGNMENT_ID,
        "--data-root", DATASET_DIR,
        "--train-samples", str(ALIGNMENT_SAMPLES),
        "--steps", str(ALIGNMENT_STEPS),
        "--accum", str(ALIGNMENT_ACCUM),
        "--max-length", str(MAX_LENGTH),
        "--pool-grid", str(POOL_GRID),
        "--stage", "alignment",
        "--parameter-precision", "fp32",
        "--learning-rate", str(ALIGNMENT_LR),
        "--scheduler", "cosine",
        "--warmup-ratio", "0.03",
        "--eval-limit", "12",
        "--save-every", "64",
        "--save-total-limit", "2",
        "--result-every", "10",
        "--skip-locked-test",
        "--tasks", "factual_caption",
    ]
    run(command, cwd=REPO, env=TRAIN_ENV)
    result_path = align_dir / "result.json"
    if not result_path.is_file():
        raise RuntimeError("Alignment result.json is missing")
    result = json.loads(result_path.read_text())
    if result.get("status") != "completed":
        raise RuntimeError("Alignment did not complete: " + result.get("error", "unknown"))
    pointer = json.loads((align_dir / "final_checkpoint.json").read_text())
    ALIGNMENT_CKPT = Path(pointer["path"])
    if not ALIGNMENT_CKPT.is_dir():
        raise RuntimeError("Alignment checkpoint is missing: " + str(ALIGNMENT_CKPT))
    audit = result.get("parameter_audit", {})
    print(json.dumps({
        "alignment_checkpoint": str(ALIGNMENT_CKPT),
        "trainable": audit.get("trainable"),
        "frozen_tensors": audit.get("frozen_tensors"),
        "final_step": result.get("update_audit", {}).get("final_step"),
        "dev_loss": result.get("after_dev", {}).get("token_weighted_loss"),
    }, indent=2))
else:
    print("Using existing alignment checkpoint:", ALIGNMENT_CKPT)

## 9. Full-parameter fine-tune

All **4,201,669,760** parameters train: Spark + SigLIP + connectors. FP32 parameter storage, BF16 AMP, AdamW8bit, no LoRA. `WALL_HOURS` bounds one Colab session; a cooperative stop still saves a complete optimizer checkpoint. If the epoch finishes, `epoch_plan_complete` is true.

In [ ]:
FULL_ID = RUN_ID + "-full"
full_dir = REPO / "results" / FULL_ID
if full_dir.exists():
    raise RuntimeError(f"Full run already exists: {full_dir}. Use a new RUN_ID.")
command = [
    PYTHON, TRAIN,
    "--run-id", FULL_ID,
    "--data-root", DATASET_DIR,
    "--formal",
    "--epochs", "1",
    "--wall-hours", str(WALL_HOURS),
    "--accum", str(FULL_ACCUM),
    "--max-length", str(MAX_LENGTH),
    "--pool-grid", str(POOL_GRID),
    "--parameter-precision", "fp32",
    "--learning-rate", str(FULL_LR),
    "--loss-balance", "none",
    "--scheduler", "cosine",
    "--warmup-ratio", "0.03",
    "--save-every", str(SAVE_EVERY),
    "--dev-every", str(DEV_EVERY),
    "--save-total-limit", "2",
    "--result-every", "10",
    "--eval-limit", str(EVAL_LIMIT),
    "--skip-locked-test",
    "--initialize-from", str(ALIGNMENT_CKPT),
]
run(command, cwd=REPO, env=TRAIN_ENV)

result_path = full_dir / "result.json"
if not result_path.is_file():
    raise RuntimeError("Full training result.json is missing")
result = json.loads(result_path.read_text())
if result.get("status") not in {"completed", "stopped"}:
    raise RuntimeError("Full training did not complete or stop cleanly: " + result.get("error", "unknown"))
pointer_path = full_dir / "final_checkpoint.json"
if not pointer_path.is_file():
    pointer_path = full_dir / "latest_checkpoint.json"
if not pointer_path.is_file():
    raise RuntimeError("No complete checkpoint pointer was written")
pointer = json.loads(pointer_path.read_text())
FINAL_CKPT = Path(pointer["path"])
if not FINAL_CKPT.is_dir():
    raise RuntimeError("Final checkpoint is missing: " + str(FINAL_CKPT))
pre_receipt = {
    "schema_version": 1, "run_id": RUN_ID,
    "source_commit": run(["git", "rev-parse", "HEAD"], cwd=REPO).stdout.strip(),
    "dataset": str(DATASET_DIR), "dataset_manifest_sha256": DATASET_MANIFEST_SHA,
    "dataset_records": manifest["record_count"], "dataset_splits": manifest["by_split"],
    "alignment_checkpoint": str(ALIGNMENT_CKPT), "final_checkpoint": str(FINAL_CKPT),
    "final_checkpoint_bytes": sum(p.stat().st_size for p in FINAL_CKPT.rglob("*") if p.is_file()),
    "training_status": result.get("status"),
    "epoch_plan_complete": result.get("summary", {}).get("epoch_plan_complete"),
    "termination_reason": result.get("termination_reason"),
    "capability_promoted": False, "test_split": "not included; no sealed-test claim",
}
(WORK / "RUN_RECEIPT.json").write_text(json.dumps(pre_receipt, indent=2) + "\n")
print(json.dumps({
    "status": result.get("status"),
    "termination_reason": result.get("termination_reason"),
    "final_checkpoint": str(FINAL_CKPT),
    "summary": result.get("summary"),
    "data_exposure": result.get("data_exposure"),
}, indent=2))

## 10. Persist the complete checkpoint

Colab local storage is not durable. With `PUSH_CHECKPOINT=True`, this uploads the complete checkpoint (model, optimizer, scheduler, RNG, exposure ledger, source binding) to the private HF model repo. The token is read from Colab Secrets or the process environment and is never written to a cell, log, or checkpoint.

This upload can take a long time; `WALL_HOURS=9` leaves headroom in a typical session. If it fails, the local checkpoint still exists for the remainder of this VM.

In [ ]:
if PUSH_CHECKPOINT:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
    if not HF_TOKEN:
        HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError("PUSH_CHECKPOINT=True but no HF_TOKEN is available. Add it to Colab Secrets.")
    remote_path = "checkpoints/" + FULL_ID
    upload_env = os.environ.copy()
    upload_env["HF_TOKEN"] = HF_TOKEN
    upload_env["HF_REPO"] = HF_CHECKPOINT_REPO
    upload_env["HF_REMOTE"] = remote_path
    upload_env["CKPT"] = str(FINAL_CKPT)
    script = (
        "import json, os; from huggingface_hub import HfApi; "
        "api=HfApi(token=os.environ['HF_TOKEN']); "
        "api.create_repo(repo_id=os.environ['HF_REPO'], repo_type='model', private=True, exist_ok=True); "
        "api.upload_folder(repo_id=os.environ['HF_REPO'], repo_type='model', folder_path=os.environ['CKPT'], "
        "path_in_repo=os.environ['HF_REMOTE'], commit_message='Spark image/frame full fine-tune ' + os.path.basename(os.environ['CKPT'])); "
        "print(json.dumps({'repo': os.environ['HF_REPO'], 'path': os.environ['HF_REMOTE'], 'uploaded': True}))"
    )
    run([PYTHON, "-c", script], cwd=REPO, env=upload_env)
    print("Uploaded to https://huggingface.co/" + HF_CHECKPOINT_REPO + "/tree/main/" + remote_path)
else:
    print("PUSH_CHECKPOINT=False; checkpoint remains local:", FINAL_CKPT)

## 11. Final receipt

This cell writes a small local receipt with the run, source, corpus, and checkpoint identities. Download or copy it before disconnecting.

In [ ]:
receipt = {
    "schema_version": 1,
    "run_id": RUN_ID,
    "source_commit": run(["git", "rev-parse", "HEAD"], cwd=REPO).stdout.strip(),
    "dataset": str(DATASET_DIR),
    "dataset_manifest_sha256": DATASET_MANIFEST_SHA,
    "dataset_records": manifest["record_count"],
    "dataset_splits": manifest["by_split"],
    "alignment_checkpoint": str(ALIGNMENT_CKPT),
    "final_checkpoint": str(FINAL_CKPT),
    "final_checkpoint_bytes": sum(p.stat().st_size for p in FINAL_CKPT.rglob("*") if p.is_file()),
    "training_status": result.get("status"),
    "epoch_plan_complete": result.get("summary", {}).get("epoch_plan_complete"),
    "termination_reason": result.get("termination_reason"),
    "capability_promoted": False,
    "test_split": "not included; no sealed-test claim",
}
path = WORK / "RUN_RECEIPT.json"
path.write_text(json.dumps(receipt, indent=2) + "\n")
print(json.dumps(receipt, indent=2))
print("Receipt:", path)